In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()
PROC_DIR = PROJECT_DIR / "Data" / "Processed"
OUT_DIR = PROJECT_DIR / "Outputs"
SUB_DIR = OUT_DIR / "Submissions"

PROC_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

print("PROC_DIR:", PROC_DIR)
print("SUB_DIR :", SUB_DIR)

PROC_DIR: /Users/linda/RiceDatathon_2026_Finance/Data/Processed
SUB_DIR : /Users/linda/RiceDatathon_2026_Finance/Outputs/Submissions


In [2]:
panel = pd.read_csv(PROC_DIR / "panel_with_amenity_scores.csv")
scoring = pd.read_csv(PROC_DIR / "scoring_with_amenity_scores.csv")

print("panel  :", panel.shape)
print("scoring:", scoring.shape)

panel  : (38941, 123)
scoring: (8997, 123)


In [3]:
TARGET_COL = "target"

# Choose ONE:
TARGET_SOURCE = "revpar_growth_2022_2025_pct"   # post
# TARGET_SOURCE = "revpar_growth_2015_2020_pct" # pre

if TARGET_SOURCE not in panel.columns:
    raise RuntimeError(f"Target column not found in panel: {TARGET_SOURCE}")

panel[TARGET_COL] = pd.to_numeric(panel[TARGET_SOURCE], errors="coerce")

In [4]:
train = panel[panel[TARGET_COL].notna()].copy()
test = scoring.copy()

print("train:", train.shape, " | target non-null:", train[TARGET_COL].notna().mean())
print("test :", test.shape)

train: (38941, 124)  | target non-null: 1.0
test : (8997, 123)


In [5]:
ID_CANDIDATES = ["ubid", "property_id"]
GROUP_COL = next((c for c in ID_CANDIDATES if c in train.columns), None)

if GROUP_COL is None:
    # Stable fields: property-level, not drivetime/time window
    stable_candidates = ["latitude", "longitude", "lat", "lon", "yearbuilt", "numunits"]
    stable = [c for c in stable_candidates if c in train.columns]
    if len(stable) == 0:
        raise RuntimeError("No ID columns and no stable columns to construct group key.")

    train["_group_key"] = train[stable].astype(str).agg("_".join, axis=1)
    test["_group_key"] = test[stable].astype(str).agg("_".join, axis=1)
    GROUP_COL = "_group_key"

print("Using group column:", GROUP_COL)

Using group column: ubid


In [6]:
BASE_FEATURES = [
    # Amenity scores
    "daily_needs_score",
    "daily_needs_score_equal",
    "lifestyle_score",
    "family_support_score",

    # Trade area
    "drivetime_min",

    # Property fundamentals
    "numunits",
    "areaperunit",
    "yearbuilt",
    "year_renov",

    # Economics / supply
    "ownrent_avg_rent",
    "ownrent_avg_mortgage",
    "ownrent_spread_abs",
    "ownrent_spread_pct",
    "supply_growth_pct",
    "supply_new_units",
    "supply_baseline_units",
]

FEATURES = [c for c in BASE_FEATURES if c in train.columns and c in test.columns]

if len(FEATURES) < 5:
    raise RuntimeError(f"Too few features found. FEATURES={FEATURES}")

print("Number of features:", len(FEATURES))
print("Features:", FEATURES)

Number of features: 16
Features: ['daily_needs_score', 'daily_needs_score_equal', 'lifestyle_score', 'family_support_score', 'drivetime_min', 'numunits', 'areaperunit', 'yearbuilt', 'year_renov', 'ownrent_avg_rent', 'ownrent_avg_mortgage', 'ownrent_spread_abs', 'ownrent_spread_pct', 'supply_growth_pct', 'supply_new_units', 'supply_baseline_units']


In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [8]:
# Install lightgbm if not already installed
import sys
!{sys.executable} -m pip install lightgbm

# Now import the required modules
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
import numpy as np

# Rest of your code remains the same
X = train[FEATURES].copy()
y = train[TARGET_COL].copy()
groups = train[GROUP_COL].astype(str).values

X_test = test[FEATURES].copy()

model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", model),
])

gkf = GroupKFold(n_splits=5)

rmse_list = []
oof = np.zeros(len(train))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    pipe.fit(X_tr, y_tr)
    pred_va = pipe.predict(X_va)

    fold_rmse = rmse(y_va, pred_va)
    rmse_list.append(fold_rmse)
    oof[va_idx] = pred_va

    print(f"Fold {fold}: RMSE = {fold_rmse:.5f}")

print("\nCV RMSE mean:", float(np.mean(rmse_list)))
print("CV RMSE std :", float(np.std(rmse_list)))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000456 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3436
[LightGBM] [Info] Number of data points in the train set: 31152, number of used features: 16
[LightGBM] [Info] Start training from score -0.051597


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1: RMSE = 0.11134
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000530 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3432
[LightGBM] [Info] Number of data points in the train set: 31153, number of used features: 16
[LightGBM] [Info] Start training from score -0.050948


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2: RMSE = 0.12861
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000534 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3435
[LightGBM] [Info] Number of data points in the train set: 31153, number of used features: 16
[LightGBM] [Info] Start training from score -0.051077


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3: RMSE = 0.12620
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000602 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3436
[LightGBM] [Info] Number of data points in the train set: 31153, number of used features: 16
[LightGBM] [Info] Start training from score -0.051116


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4: RMSE = 0.11415
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000645 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3431
[LightGBM] [Info] Number of data points in the train set: 31153, number of used features: 16
[LightGBM] [Info] Start training from score -0.054203
Fold 5: RMSE = 0.14375

CV RMSE mean: 0.1248107991460928
CV RMSE std : 0.011576557283109268


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [9]:
pipe.fit(X, y)
pred_test = pipe.predict(X_test)

test["pred_target_lgbm"] = pred_test
print(test["pred_target_lgbm"].describe())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000778 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3438
[LightGBM] [Info] Number of data points in the train set: 38941, number of used features: 16
[LightGBM] [Info] Start training from score -0.051788
count    8997.000000
mean       -0.053032
std         0.082543
min        -0.444786
25%        -0.102441
50%        -0.060324
75%        -0.011223
max         0.899759
Name: pred_target_lgbm, dtype: float64


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [10]:
ID_CANDIDATES = ["row_id", "id", "property_id", "ubid"]
ID_COL = next((c for c in ID_CANDIDATES if c in scoring.columns), None)

if ID_COL is None:
    submission = test[["pred_target_lgbm"]].copy()
else:
    submission = test[[ID_COL, "pred_target_lgbm"]].copy()

SUB_PATH = SUB_DIR / "submission_lgbm.csv"
submission.to_csv(SUB_PATH, index=False)

print("Saved submission:", SUB_PATH)
print(submission.head())

Saved submission: /Users/linda/RiceDatathon_2026_Finance/Outputs/Submissions/submission_lgbm.csv
                       ubid  pred_target_lgbm
0  76X6WHJM+WGJ-18-14-18-15         -0.012966
1  865523H8+8QC-18-14-18-14         -0.065176
2  865538F3+9FQ-18-14-18-15         -0.145550
3  865QRR26+P69-18-14-18-15         -0.065071
4  862476FP+H6F-18-14-18-15         -0.147347


In [11]:
train_out = train.copy()
train_out["oof_pred_lgbm"] = oof
train_out.to_csv(PROC_DIR / "train_with_oof_pred.csv", index=False)

print("Saved OOF predictions to:", PROC_DIR / "train_with_oof_pred.csv")

Saved OOF predictions to: /Users/linda/RiceDatathon_2026_Finance/Data/Processed/train_with_oof_pred.csv
